In [24]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import load_and_clean_dataset

DATASET_PATH = PROJECT_ROOT / "data" / "bangla_news.csv"

df_clean = load_and_clean_dataset(DATASET_PATH)

print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (11844, 6)


Bengali Text Preprocessing

In [25]:
sample_text = (
    "নাদাল-জোকোভিচ মহারণ ঘিরে টেনিসপ্রেমীদের অনেক আগ্রহ ছিল। "
    "অলিম্পিকে পুরুষ এককের দ্বিতীয় রাউন্ডে জোকোভিচের কাছে "
    "হেরেছেন নাদাল।"
)

print(sample_text)
print("Character count:", len(sample_text))

নাদাল-জোকোভিচ মহারণ ঘিরে টেনিসপ্রেমীদের অনেক আগ্রহ ছিল। অলিম্পিকে পুরুষ এককের দ্বিতীয় রাউন্ডে জোকোভিচের কাছে হেরেছেন নাদাল।
Character count: 124


In [26]:
import re

baseline_tokens = re.findall(r"[\u0980-\u09FF]+", sample_text)

print(baseline_tokens)
print("Number of tokens:", len(baseline_tokens))

['নাদাল', 'জোকোভিচ', 'মহারণ', 'ঘিরে', 'টেনিসপ্রেমীদের', 'অনেক', 'আগ্রহ', 'ছিল', 'অলিম্পিকে', 'পুরুষ', 'এককের', 'দ্বিতীয়', 'রাউন্ডে', 'জোকোভিচের', 'কাছে', 'হেরেছেন', 'নাদাল']
Number of tokens: 17


In [27]:
from bnlp import BasicTokenizer

tokenizer = BasicTokenizer()

mixed_text = "নাদাল-জোকোভিচ ম্যাচে Djokovic 6-1, 6-4 সেটে জিতেছেন। " "US Open 2024!"

tokens = tokenizer(mixed_text)

print(tokens)
print("Number of tokens:", len(tokens))

['নাদাল', '-', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', '6', '-', '1', ',', '6', '-', '4', 'সেটে', 'জিতেছেন', '।', 'US', 'Open', '2024', '!']
Number of tokens: 19


In [28]:
import re
import string

print("Punctuation:", string.punctuation)

cleaned_tokens = [
    token for token in tokens if token not in string.punctuation and token != "।"
]

print("\nTokens after punctuation removal:")
print(cleaned_tokens)
print("Number of tokens:", len(cleaned_tokens))

Punctuation: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

Tokens after punctuation removal:
['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', '6', '1', '6', '4', 'সেটে', 'জিতেছেন', 'US', 'Open', '2024']
Number of tokens: 13


In [29]:
cleaned_tokens_no_numbers = [
    token for token in cleaned_tokens if not re.fullmatch(r"\d+", token)
]

print("Tokens after removing standalone numbers:")
print(cleaned_tokens_no_numbers)
print("Number of tokens:", len(cleaned_tokens_no_numbers))

Tokens after removing standalone numbers:
['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', 'সেটে', 'জিতেছেন', 'US', 'Open']
Number of tokens: 8


In [30]:
def preprocess_text(text):
    # Handle missing or invalid input
    if not isinstance(text, str):
        return []

    # Step 1: Tokenize Bengali and mixed-language text
    tokens = tokenizer(text)

    # Step 2: Remove punctuations
    cleaned_tokens = [
        token for token in tokens
        if not all(
            char in string.punctuation or char in "।–—…"
            for char in token
        )
    ]

    # Step 3: Remove standalone numeric tokens
    cleaned_tokens = [
        token for token in cleaned_tokens
        if not re.fullmatch(r"\d+", token)
    ]

    return cleaned_tokens

In [31]:
processed_sample = preprocess_text(mixed_text)

print(processed_sample)
print("Number of tokens:", len(processed_sample))

['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', 'সেটে', 'জিতেছেন', 'US', 'Open']
Number of tokens: 8


In [32]:
print(preprocess_text(""))
print(preprocess_text(None))

[]
[]


In [33]:
print("Normal text:", preprocess_text(mixed_text))
print("Empty string:", preprocess_text(""))
print("Missing value:", preprocess_text(None))
print("Number:", preprocess_text(2024))

Normal text: ['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', 'সেটে', 'জিতেছেন', 'US', 'Open']
Empty string: []
Missing value: []
Number: []


Preprocessing

In [34]:
for index in df_clean.index[:5]:
    article = df_clean.loc[index, "content"]
    processed = preprocess_text(article)

    print("=" * 80)
    print("Article index:", index)
    print("\nOriginal text:")
    print(article[:300])
    print("\nProcessed tokens:")
    print(processed[:30])
    print("Total tokens:", len(processed))

Article index: 0

Original text:
ছবি: সংগৃহীত সিন নদীতে অলিম্পিকের উদ্বোধনী অনুষ্ঠান হয়েছিল। এই নদীর পানি একেবারে দূষণমুক্ত, তা প্রমাণে প্যারিসের মেয়র নিজেই পানিতে নেমে পড়েছিলেন। কিন্তু এত কিছুর পরও সিন নদীকে দূষণমুক্ত প্রমাণ করা গেল না। দূষণের কারণে সিন নদীতে অনুষ্ঠিতব্য অলিম্পিকের ট্রায়াথলন ইভেন্ট স্থগিত করা হয়েছে। খবর দ্য গ

Processed tokens:
['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'হয়েছিল', 'এই', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'তা', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'নিজেই', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিন্তু', 'এত', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'করা']
Total tokens: 90
Article index: 1

Original text:
২০২৫ সালের জানুয়ারিতে এশিয়ান ক্রিকেট কাউন্সিল (এসিসির) চেয়ারম্যানের পদের মেয়াদ শেষ হচ্ছে। এরই মধ্যে জল্পনা কল্পনা চলছে এসিসির চেয়ারম্যান কে হচ্ছে। এর মধ্যে সবচেয়ে বেশি শোনা যাচ্ছে মহসিন নাকভির নাম। পাকিস্তানের বেশ কয়েকটি সংবাদ মাধ্যম জানিয়েছে পিসিবির বর্তমান চেয়ারম্যানই হচ্ছেন এসিসির নতুন স

Processe